# Tutorial

**Section contents**

In this section, we introduce methods and functionalities of the dicomhandler package. Its goal is to *handle*, *convert* and *report* data summaries about different DICOM
files related to radiation therapy. DICOM (Digital Imaging and Communications in Medicine) is a format that is used for viewing, storing, sharing and retrieving medical images of a patient. The medical images can have
different modalities, such as:

1. RT Dose (Radiotherapy Dose) : Contains the patient medical images that describe 2D or 3D radiation dose data, generated from treatment planning systems or similar devices. This modality help the medical staff to view the old radiation dose data and storing the new ones. 

2. RT Plan (Radiotherapy Plan) : Contains the patient medical images related to the treatment plans. These images help the medical staff to control if the requirements for transferring radiation therapy plans between devices are satisfied. The images are related, for example, to the positions of the gantry leaves during the radiotherapy treatment.


3. RT Structure (Radiotherapy Structure) : Contains the patient medical images of the structures of interest (body parts and metastasis) and related data. This modality includes contour data for the regions of interest (ROIs) of the structures. These contour data are relevant to radiation treatment planning. 

There are types of DICOM files that are not treated. A future work is to add methods and funcionalities even for other types. We can see a complete list [here](https://dicom.innolitics.com/ciods).

The core of our package is the class *DicomInfo*. With it, we can storing the three DICOM files described above. The goal of this class is to *handle* and *convert* the files RT Plan and RT Structure, not RT Dose (future work). The method *report*, that is not contained in the class *DicomInfo*, is used for storing and comparing information about different RT Structure files. 
The datasets folder, that contains the DICOM structure files and DICOM plan files of the patients, is in the directory package folder -> docs -> source -> tutorial -> datasets. The tutorial notebook (with extension ".ipynb"), that can be used for playing with the package, is in the package folder -> docs -> source -> tutorial -> basic -> tutorial.ipynb 

## Class DicomInfo
Now we describe how this class stores, handles and converts the three DICOM files.  

***Storing DICOM***  
We read the DICOM file before storing it in a instance of DicomInfo Class.
The first thing to do, before storing a DICOM file, is reading it. 
In order to read it, we import the package *os* (used for retrieving the path in which are the DICOM files) and the package *pydicom* (used for reading a DICOM file). The extension of a DICOM file is ".dcm" . The variable *patient_dicom_struct* contains the RT Structure of a patient. The type of this variable is *pydicom.dataset.FileDataset* . We can print the name of the patient that corresponds 

In [1]:
import os
import pydicom
path = os.getcwd() 
dicom_structure_path = str(path) + '/datasets/structure/RS_0.dcm'
patient_dicom_struct = pydicom.dcmread(dicom_structure_path)

Now, we can create a instance of DicomInfo Class. The name of this instance is *di*. For creating a instance of the class, we need at least one of the three DICOM files described at the beginning of the section. We import the DicomInfo Class from the dicomhandler folder. The class is inside the dicom_info.py file:

In [2]:
from dicomhandler.dicom_info import Dicominfo
di = Dicominfo(patient_dicom_struct)

At this point, *di* stores the RT Structure. For example, we can retrieve the name of the Patient:

In [3]:
di.dicom_struct.PatientName

'ESCUTI^FRANCISCA LILIANA'

and the name of the second structure of this patient:

In [4]:
di.dicom_struct.StructureSetROISequence[1].ROIName

'5 PTV +1.0 mm'

We can start to see the methods that *handling* and *converting* the DICOM files. 
The methods that handle this type of DICOM file are **anonymize**, **rotate**, **translate** and **add_margin**. All these methods can change the coordinates of the patient's structures.
The object *di.dicom_struct.StructureSetROISequence* contains all the names of the patient's structures. This object is of the type *pydicom.sequence.Sequence*. Each element of this sequence is a *pydicom.dataset.Dataset* pydicom object. 
We can extract, for example, the name of the patient's third structures. The third structure has index 2 inside the sequence:

In [5]:
di.dicom_struct.StructureSetROISequence[2].ROIName

'Hippocampus Righ'

The object *di.dicom_struct.ROIContourSequence* contains all the coordinates of the patient's structures. This object is of the type *pydicom.sequence.Sequence*. Each element of this sequence is a *pydicom.dataset.Dataset*, so each structure's coordinates are contained in this type of pydicom object. We can see a structure as a 3D object, and so each structure is represented by a certain number of slices in 2D.
Each element of the *di.dicom_struct.ROIContourSequence* has a attribute called *ContourSequence*. This attribute is a *pydicom.sequence.Sequence* as well. Each element of this sequence is a *pydicom.dataset.Dataset*, and contain the contour data a specific slice. 
For example, we can extract the second structure's coordinates of the second slice (the command is commented for graphic purposes).

In [6]:
# di.dicom_struct.ROIContourSequence[1].ContourSequence[1].ContourData 

The contour data of a slice is a sequence of points in three dimension represented in the format [x0, y0, z0, x1, y1, z1, ...]. For each slice, the z-dimension is fixed. 

***Handling DICOM***

With the method **anonymize**, we can anonymize the private information of the Patient such as the name of the patient, the birthday, the operator name that made the radiotherapy and the creation date of the file (that correspond to the date of the treatment). For each of these attributes, we can decide which one to anonymize setting it to True or False.
For example, we can anonymize only the name of the Patient. We can see the name of the patient before and after the anonymization:

In [7]:
di_anony = di.anonymize(name=True, birth=False, operator=False, creation=False)
# print(di.dicom_struct.PatientName) 
# print(di_anony.dicom_struct.PatientName)

The method **rotate** is used for making a rotation pitch, yaw or roll of a structure. The following image represent the three types of the rotation for a airplane:
![alt text for screen readers](images/axes.png "Text to show on mouseover")

The instance **di_rotate** will contain the rotated coordinates of the second structure. We can print the rotated structure's coordinates of the first slice (the printing command is commented for graphic purposes). By default,
rotations are performed for the isocentre (that is the ultimate structure of the sequence *di.dicom_struct.ROIContourSequence*) or an arbitrary point defined by the user. 
For example, we want to make a roll rotation of the patient’s second structure by 200 degrees.

In [8]:
struct_name = di.dicom_struct.StructureSetROISequence[1].ROIName
di_rotate = di.rotate(struct_name, 200.0, 'roll')
# di_rotate.dicom_struct.ROIContourSequence[1].ContourSequence[1].ContourData 

The method **translate** is used for translating a structure along the axes x, y or z. By default,
translations are performed for the isocentre (that is the ultimate structure of the sequence *di.dicom_struct.ROIContourSequence*) or an arbitrary point defined by the user. For example, we want to make a translation by 2 mm of the second structure along x: 

In [9]:
struct_name = di.dicom_struct.StructureSetROISequence[1].ROIName         
di_translate = di.translate(struct_name, 2.0, 'x')

With the method **add_margin**, we can increase/decrease the margin of a specific structure of the patient. The increasing/decreasing of a structure is in mm. We can see the coordinates of original and increased structure's first slice (the two commands are commented for graphic purposes).
For example, we can increase the patient’s second structure by 2 mm:

In [ ]:
struct_name = di.dicom_struct.StructureSetROISequence[1].ROIName
di_increased = di.add_margin(struct_name, 2.0) 

As we have seen, the extraction of the contour data from DicomInfo object is a bit trivial. The DicomInfo instance *di* is created using a RT Structure *patient_dicom_struct*. All the information of the RT Structure are inside the *di*:

In [11]:
di.dicom_struct.ROIContourSequence[1].ContourSequence[1].ContourData == \
patient_dicom_struct.ROIContourSequence[1].ContourSequence[1].ContourData 

True

In [12]:
di.dicom_struct.PatientName == patient_dicom_struct.PatientName

True

For simplifying the extraction of information from RT Structure and RT Plan, we use the methods **structure_to_excel**, **mlc_to_excel**, **areas_to_dataframe** and **info_to_dataframe** of the Class *DicomInfo*. With these methods, we want to give the user a way to better structure the information contained in RT Structure and RT Plan. Now, we want to instance a DicomInfo object which also contains RT Plan. The RT Structure and RT Plan must refer to the same patient. The pydicom object *patient_dicom_plan* contains the information of RT Plan.

In [13]:
path = os.getcwd() 
dicom_structure_path = str(path) + '/datasets/structure/RS_0.dcm'
patient_dicom_struct = pydicom.dcmread(dicom_structure_path)
dicom_plan_path = str(path) + '/datasets/plan/RP_0.dcm'
patient_dicom_plan = pydicom.dcmread(dicom_plan_path)
di = Dicominfo(patient_dicom_struct, patient_dicom_plan)

Now, *di* stores the information that were in *patient_dicom_struct* and *patient_dicom_plan*

The method **structure_to_excel** extracts the information of the cartesian coordinates (relative positions) for all or some structures. The output file provides the coordinates of each structure in its own sheet. For example, we want to extract this information for the third structure. The method return a file in the format .xlsx:

In [19]:
struct_name_1 = di.dicom_struct.StructureSetROISequence[1].ROIName
struct_name_2 = di.dicom_struct.StructureSetROISequence[2].ROIName
di.structure_to_excel('name_file', names = [struct_name_1, struct_name_2])

The RT Plan contains information about multilead collimator (MLC) positions, control points, gantry angles, gantry orientation and patient table angle. In order to understand better what is a MLC, we can show the next image:
![a](images/mlc2.jpg "")

The MLC create a photon beam (radiotherapy) that passes through a irregular form created by the leaves. The photom beam radiates a structure (metastasis) of the patient. The gantry is a mechanical support for mounting a device to be moved in a circular path. In the next image, we can have a better understand about gantry, MLC, isocenter and patient table: ![b](images/gantry.png "")

During a radiotherapy, a lot of information about the gantry's movements of the gantry and leaves of MLC are stored in the RT Plan. During a treatment, we have a lot of differents movements of the different supports. For each movement, we have some control points in which we save the information about the multilead collimator leaves positions, gantry angles, gantry orientation and patient table angle.
The method **mlc_to_excel** extracts these information. It returns a file with extension .xlsx . 

In [15]:
di.mlc_to_excel('name_file')

The method **areas_to_dataframe** extract the areas of the irregular forms created by the leaves during a treatement. It returns a pandas dataframe.

In [16]:
df_areas = di.areas_to_dataframe()  

The method **info_to_dataframe** extract information about the RT Plan and RT Structure of a patient.
It returns all the information about the metastasis/body structures that are contained in the RT Plan. These information are prescribed dose, reference points, dose to references points, maximum, minimun and mean radius, the center of mass and distance to isocenter. This method search the information in the RT Structure that correspond to the structures in the RT Plan. Tipically, the names of the structures in RT PLan and RT Structure are different even if they refer to the same structure. For example, the strcuture named "3 GTV" in the RT Structure, could have the corresponding information in the RT Plan but in this DICOM file the structure is named "3 GTV + 1 mm". 


In [17]:
df_struct_plan = di.info_to_dataframe()

/home/nicola/dise/lib/python3.10/site-packages/dicomhandler/dicom_info.py:795: UserWarning: It is not guaranteed that for each element in the                 pydicom struct                there is the corresponding element in the pydicom plan
  warnings.warn(


## Method report

The method **report** makes a comparision between the DICOM Structure of two distinct patients, about a specific structure. For example, about the third structure:


In [18]:
from  dicomhandler.dicom_info import Dicominfo
from dicomhandler.report import report
import pydicom

path = os.getcwd() 
dicom_structure_1_path = str(path) + '/datasets/structure/RS_0.dcm' 
dicom_structure_2_path = str(path) + '/datasets/structure/RS_1.dcm' 
patient_struct_1 = pydicom.dcmread(dicom_structure_1_path)
patient_struct_2 = pydicom.dcmread(dicom_structure_2_path)
struct_name = di.dicom_struct.StructureSetROISequence[2].ROIName
di_1 = Dicominfo(patient_struct_1)
di_2 = Dicominfo(patient_struct_2)
report(di_1, di_2, struct_name)

,Parameter,Value [mm]
0,Max radius,21.828
1,Min radius,0.704
2,Mean radius,12.412
3,STD radius,4.775
4,Variance radius,22.802
5,Max distance,0.000
6,Min distance,0.000
7,Mean distance,0.000
8,STD distance,0.000
9,Variance distance,0.000
